# Phase 6c Wave 4: Black Holes as Quantum Error-Correcting Codes

**A non-specialist tour of Hayden-Preskill scrambling, code distance, and why anyons matter for black-hole information.**

Companion notebook to `papers/paper35_qec_holography/paper_draft.tex`.

## The puzzle

When you throw a book into a fire, the information is scrambled — but in principle recoverable. The fire's microstates encode every word; you'd just need to decode an absurd amount of thermal radiation.

When you throw a book into a black hole, classical general relativity says the information is gone forever. Quantum mechanics says it can't be, because evolution is unitary. Hawking's 1976 calculation made the conflict sharp: thermal radiation seems to carry no information about what fell in.

**Hayden and Preskill (2007) resolved a key piece.** If the black hole is *old* (mostly evaporated), and an external partner has been collecting Hawking radiation, then a new throw-in bit can be *recovered* in $O(\log S_{BH})$ time — a *logarithmic* scrambling time, much faster than the black hole's evaporation. The black hole acts as a quantum mirror.

## The QEC reframe

More recent work (Almheiri-Dong-Harlow 2015; Pastawski-Yoshida-Harlow-Preskill 2015) reframed the AdS/CFT bulk as a quantum *error-correcting code*: the bulk encodes redundant copies of the boundary information, and the horizon is the code's protective layer. Yoshida-Kitaev (2017) gave an explicit decoder.

## What this paper adds

In the project's emergent-gravity framework, the horizon is equipped with a *modular tensor category* (MTC) of anyon-like excitations. This paper formalizes the Hayden-Preskill protocol on top of that MTC substrate and proves a structural identity: **a non-abelian anyon spectrum is necessary AND sufficient for non-trivial code distance, which automatically gives non-trivial scrambling time.** Topologically rich = quantum-info-rich. The trivial (abelian-only) substrate has zero code distance and zero scrambling — it's not a quantum-error-correcting code at all.

## 1. The two observables

We'll work with two numbers per MTC substrate:

- **Code distance** $d_C := \log d_{\max}$, where $d_{\max}$ is the largest quantum dimension of any anyon. Heuristically: how much "topological room" the substrate gives the encoded qubit to hide from local errors.
- **Scrambling-time bound** $t_{\rm scr} := \log D^2$, where $D^2 := \sum_a d_a^2$ is the total quantum dimension squared. Heuristically: the substrate's contribution to the time it takes information to spread across all degrees of freedom.

Both are dimensionless (logs of pure numbers). Both vanish when the MTC has only one anyon (the vacuum) — i.e., in the abelian-trivial limit.

In [ ]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.qec_holography import (
    FIBONACCI_SPECTRUM, ISING_SPECTRUM,
    SU3K2_SPECTRUM, TRIVIAL_ABELIAN_SPECTRUM,
    code_distance, scrambling_time_bound,
    code_distance_admissible, global_dim_sq,
)
from src.core.visualizations import COLORS, fig_code_distance_vs_fusion_spectrum

import math
phi = (1 + math.sqrt(5)) / 2
print(f'  Anchor: φ (golden ratio)         = {phi:.6f}')
print(f'  log φ                            = {math.log(phi):.6f}      ← Fibonacci code distance d_C')
print(f'  log √2                           = {math.log(math.sqrt(2)):.6f}      ← Ising code distance d_C (smaller than Fibonacci!)')
print(f'  log 2                            = {math.log(2):.6f}      ← reference: smallest qubit code distance')
print(f'  → Fibonacci is the smallest non-abelian MTC SUFFICIENT FOR UNIVERSAL QC')
print(f'    (Nayak-Simon-Stern-Freedman-Das Sarma 2008 §3); Ising has smaller d_C')
print(f'    but its braiding alone is not universal.')

## 2. Four substrates, in plain language

**Fibonacci anyons.** A two-anyon MTC: vacuum and "$\tau$" with quantum dimension $\varphi$ (the golden ratio $\approx 1.618$). The smallest non-abelian MTC for which fault-tolerant universal quantum computation is known. Topologically very rich.

**Ising anyons.** A three-anyon MTC: vacuum, $\sigma$ (with $d_\sigma = \sqrt{2}$), and a fermion $\psi$. Realized in chiral $p$-wave superconductors and the Moore-Read fractional quantum Hall state. Non-abelian, and *its* code distance is actually *smaller* than Fibonacci's (it's the smallest admissible-by-d_C MTC); but Ising braiding alone doesn't give universal QC.

**SU(3)$_{k=2}$ Fibonacci sub-sector.** A 2-element truncation of an SU(3) Wess-Zumino-Witten model that produces the same anyon spectrum as Fibonacci ($d_{\max} = \varphi$, $D^2 = 1+\varphi^2$). Cross-check: independent path to the same observables. *Note*: this is a *sub-sector*, not the full SU(3)$_{k=2}$ MTC (which has 6 simple objects).

**Trivial abelian.** Just the vacuum. $d_{\max} = 1$, $D^2 = 1$, $d_C = t_{\rm scr} = 0$. The substrate has *no* code distance and *no* scrambling time. It is, by the structural theorem, NOT a quantum-error-correcting code on this protocol.

In [ ]:
labels = [
    ('Fibonacci',                  FIBONACCI_SPECTRUM,    'τ has d=φ; smallest non-abelian for universal QC'),
    ('Ising',                      ISING_SPECTRUM,        'σ has d=√2; chiral p-wave SC, not universal alone'),
    ('SU(3)_{k=2} Fib sub-sector', SU3K2_SPECTRUM,        'truncated 2-element sub-sector, cross-check'),
    ('Trivial-Ab',                 TRIVIAL_ABELIAN_SPECTRUM, 'singleton vacuum (FALSIFIER)'),
]
header = f'  {"name":28s} | {"d_max":>6} | {"D²":>6} | {"d_C":>6} | {"t_scr":>6} | admissible | description'
print(header)
print('  ' + '-' * (len(header) - 2))
for name, sp, desc in labels:
    print(f'  {name:28s} | {sp.d_max:6.3f} | {global_dim_sq(sp):6.3f} | '
          f'{code_distance(sp):6.3f} | {scrambling_time_bound(sp):6.3f} | '
          f'{str(code_distance_admissible(sp)):>10s} | {desc}')

## 3. The structural identity

**Claim (formalized as a Lean biconditional):**
$$\text{code distance } d_C > 0 \iff \exists \text{ a non-abelian anyon } (d_a > 1).$$
Plus a forward implication: $d_C > 0 \Rightarrow t_{\rm scr} > 0$.

**In words:** A substrate has non-trivial code distance *if and only if* it has at least one anyon with non-trivial quantum dimension. And whenever it does, the scrambling-time bound is automatically positive — the inequality $D^2 \geq d_{\max}^2$ does the work.

**Why this is non-trivial.** It's natural to imagine that code distance and scrambling time are independent observables: maybe a substrate is rich in fusion channels (large $D^2$) but each individual anyon has $d = 1$ (no large $d_{\max}$). The structural identity forecloses this: any positive code distance forces positive scrambling time. The two observables move together.

In [3]:
for name, sp, _ in labels:
    has_nonabelian = any(d > 1.0 for d in sp.quantum_dims)
    d_C_positive = code_distance(sp) > 0
    t_scr_positive = scrambling_time_bound(sp) > 0
    print(f'  {name:12s}: ∃ non-abelian = {has_nonabelian}    d_C > 0: {d_C_positive}    t_scr > 0: {t_scr_positive}')
print()
print('  → biconditional: "∃ non-abelian" ↔ "d_C > 0" matches in every row.')
print('  → forward implication: "d_C > 0" ⇒ "t_scr > 0" matches in every row.')

  Fibonacci   : ∃ non-abelian = True    d_C > 0: True    t_scr > 0: True
  Ising       : ∃ non-abelian = True    d_C > 0: True    t_scr > 0: True
  SU(3)_{k=2} : ∃ non-abelian = True    d_C > 0: True    t_scr > 0: True
  Trivial-Ab  : ∃ non-abelian = False    d_C > 0: False    t_scr > 0: False

  → biconditional: "∃ non-abelian" ↔ "d_C > 0" matches in every row.
  → forward implication: "d_C > 0" ⇒ "t_scr > 0" matches in every row.


## 4. Universal recovery at the Hayden-Preskill bound

The Hayden-Preskill protocol asks: can a partner who collects Hawking radiation recover Alice's encoded qubit, given the substrate's scrambling-time bound? The Lean theorem `recovery_at_scrambling_bound` says **yes, the threshold inequality $\log d_a \leq \log D^2 = t_{\rm scr}$ holds universally — for any encoding choice.**

The proof is delightfully short: case-split on whether the encoding anyon's quantum dimension $d_a$ is $\geq 1$ or $< 1$. If $d_a \geq 1$, then $\log d_a \leq \log d_a^2 \leq \log D^2 = t_{\rm scr}$ (using $D^2 \geq d_a^2$). If $d_a < 1$, then $\log d_a \leq 0 \leq t_{\rm scr}$. Either way, recovery threshold $\leq$ scrambling bound.

**Honest scope:** "recovery is possible at the bound" is a *threshold inequality* between substrate-only quantities; we do **not** construct the Yoshida-Kitaev decoder, do not formalize the explicit decoding circuit, and do not prove a concrete recovery probability. The Lean theorem encodes the substrate-side condition — the matching information-theoretic decoder construction is deferred per the Phase 6c roadmap.

## 5. Visualization

**Left panel.** Four bars showing code distance $d_C$ across substrates. Three substrates clear the admissibility threshold (zero); trivial-abelian sits at exactly zero (visible as a flat bar). The dotted reference line at $\log 2 \approx 0.693$ shows that Fibonacci and SU(3)$_{k=2}$ are *strictly below* the smallest power-of-2 quantum-dimension reference — confirming Fibonacci as the minimal admissible MTC.

**Right panel.** Same four bars for the scrambling-time bound $t_{\rm scr}$. Same admissibility pattern: three substrates positive, trivial-abelian zero. The graphical signature of the correctness-push.

In [4]:
# viz-ref: fig_code_distance_vs_fusion_spectrum
fig_code_distance_vs_fusion_spectrum().show()

## 6. Honest scope

**What this work proves.** Given the Phase 6a Wave 3 horizon-MTC substrate, the Hayden-Preskill information-theoretic observables (code distance, scrambling time) are tied together by a clean structural biconditional — non-abelian anyon richness is necessary and sufficient. The biconditional is a 10-theorem Lean module with zero `sorry` and zero new axioms.

**What this work does not prove.**

1. *That nature has a non-abelian horizon MTC.* This is an assumption inherited from Phase 6a Wave 3 (`H_HorizonBoundaryCondition.areaLeading`), itself a tracked external hypothesis. If the horizon is somehow abelian-only at the microscopic level, Hayden-Preskill recovery as formulated here doesn't apply.
2. *AdS/CFT spectrum identification.* We do not match the MTC to a specific holographic CFT — that is deferred per the Phase 6c roadmap.
3. *The Yoshida-Kitaev decoder.* We assert recovery is possible at the bound but do not construct the explicit decoding circuit.
4. *Page-curve quantitative reproduction.* The $S_{BH}$ area-law portion is not addressed here; this is the *substrate-only* contribution.

**What would falsify the bridge.** Any substrate-level theory in which a non-abelian anyon spectrum exists *and yet* code distance and scrambling time are decoupled — that would invalidate the biconditional. We've shown such a substrate cannot satisfy the formalization's structural assumptions; observation of one in nature would force a revision.

## 7. Where to go next

- **Paper:** `papers/paper35_qec_holography/paper_draft.tex` — full Hayden-Preskill formalization.
- **Lean module:** `lean/SKEFTHawking/QECHolographyBridge.lean` — 10 theorems.
- **Companion technical notebook:** `Phase6c4_QECHolography_Technical.ipynb` — paper-section walkthrough with Lean theorem inventory.
- **Phase 6a W3 upstream:** `lean/SKEFTHawking/BHEntropyMicroscopic.lean` — `H_HorizonBoundaryCondition` and the Kaul-Majumdar microscopic entropy. Phase 6c Wave 5 (the RT/CH note) extends the same substrate.